In [1]:
!pip install -q paddlepaddle
!pip install -q paddle2onnx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 MB 8.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 3.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.6 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 44.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 89.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 453.5/453.5 kB 30.6 MB/s eta 0:00:00


In [3]:
import subprocess, sys, os, glob
 
# 1. Siapkan folder output
os.makedirs("/kaggle/working/onnx_models", exist_ok=True)

# 2. Fungsi konversi ONNX (Tanpa argumen --input_shape_dict yang membuat crash)
def convert_to_onnx(model_dir, save_path):
    print(f"Mengonversi {model_dir}...")
    cmd = [
        "paddle2onnx",
        "--model_dir", str(model_dir),
        "--model_filename", "inference.json",
        "--params_filename", "inference.pdiparams",
        "--save_file", str(save_path),
        "--opset_version", "11",
        "--enable_onnx_checker", "True"
    ]
    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode == 0:
        print(f"✓ Berhasil: {save_path}")
    else:
        print(f"✗ Gagal: {res.stderr}")

# 3. Eksekusi Konversi
base_path = "/kaggle/input/notebooks/herdinthorikn/finetune-paddleocr/output"

convert_to_onnx(f"{base_path}/det_infer", "/kaggle/working/onnx_models/hematin_detector.onnx")
convert_to_onnx(f"{base_path}/rec_infer", "/kaggle/working/onnx_models/hematin_recognition.onnx")

# 4. Verifikasi hasil
print("\n--- HASIL KONVERSI ONNX ---")
!ls -lh /kaggle/working/onnx_models

Mengonversi /kaggle/input/notebooks/herdinthorikn/finetune-paddleocr/output/det_infer...
✓ Berhasil: /kaggle/working/onnx_models/hematin_detector.onnx
Mengonversi /kaggle/input/notebooks/herdinthorikn/finetune-paddleocr/output/rec_infer...
✓ Berhasil: /kaggle/working/onnx_models/hematin_recognition.onnx

--- HASIL KONVERSI ONNX ---
total 77M
-rw-r--r-- 1 root root 4.7M Jun  4 04:06 hematin_detector.onnx
-rw-r--r-- 1 root root  73M Jun  4 04:06 hematin_recognition.onnx


In [4]:
#Verifikasi hasil

import os
import shutil

print("Mencari kamus karakter (rec_char_dict.txt)")
ditemukan = False

for root, dirs, files in os.walk('/kaggle/input/notebooks/herdinthorikn/finetune-paddleocr/configs'):
    if 'rec_char_dict.txt' in files:
        source_path = os.path.join(root, 'rec_char_dict.txt')
        dest_path = '/kaggle/working/onnx_models/rec_char_dict.txt'
        
        # Salin file ke markas ONNX kita
        shutil.copy(source_path, dest_path)
        
        print(f"DICT DITEMUKAN DI: {source_path}")
        print(f"Berhasil disalin ke: {dest_path}")
        ditemukan = True
        break # Hentikan pencarian setelah ketemu

if not ditemukan:
    print("DICT TIDAK DITEMUKAN")

print("\n--- ISI FOLDER ONNX ---")
!ls -lh /kaggle/working/onnx_models

Mencari kamus karakter (rec_char_dict.txt)
DICT DITEMUKAN DI: /kaggle/input/notebooks/herdinthorikn/finetune-paddleocr/configs/rec_char_dict.txt
Berhasil disalin ke: /kaggle/working/onnx_models/rec_char_dict.txt

--- ISI FOLDER ONNX ---
total 77M
-rw-r--r-- 1 root root 4.7M Jun  4 04:06 hematin_detector.onnx
-rw-r--r-- 1 root root  73M Jun  4 04:06 hematin_recognition.onnx
-rw-r--r-- 1 root root  176 Jun  4 04:06 rec_char_dict.txt


In [5]:
#ZIP hasi;
import shutil, os
 
out_zip = "/kaggle/working/hematin_onnx_models"
shutil.make_archive(out_zip, "zip", "/kaggle/working/onnx_models")
 
size_mb = os.path.getsize(f"{out_zip}.zip") / 1024 / 1024
print(f" {out_zip}.zip ({size_mb:.1f} MB)")
print("\nModel berhasil di ZIP")

 /kaggle/working/hematin_onnx_models.zip (70.1 MB)

Model berhasil di ZIP
